# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset describing protein abundance and modification from mass spectrometry of human mast cell extracellular vesicles, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema available at:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading
We load dataset metadata and content using `mlcroissant`. This will let us discover available record sets and their structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant package
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', None)}\n{getattr(metadata, 'description', None)}")

## 2. Data Overview
Let's list all available `RecordSet` entities using their `@id`s. For each record set, we show the available fields and columns (also by `@id`).
This helps us understand the contents and structure we can work with.


In [ ]:
# This code prints detailed info (by @id) for all RecordSets and their Fields/Columns.
from pprint import pprint

# Helper for getting a dict of RecordSets by @id
def get_recordsets(meta):
    if hasattr(meta, 'record_set') and meta.record_set:
        rs = meta.record_set
        if isinstance(rs, dict):
            return {rs['@id']: rs}
        elif isinstance(rs, list):
            return {x['@id']: x for x in rs}
    # fallback for some older spec versions
    if hasattr(meta, 'recordSets') and meta.recordSets:
        rs = meta.recordSets
        if isinstance(rs, dict):
            return {rs['@id']: rs}
        elif isinstance(rs, list):
            return {x['@id']: x for x in rs}
    return {}

recordsets = get_recordsets(metadata)
if not recordsets:
    print("No top-level record sets stated; will attempt to infer by reading records.")
    # Try to infer: by running dataset.records() we get 'record_set' keys
    recordset_ids = set()
    for recset in dataset.list_record_sets():
        print(f"Discovered RecordSet @id: {recset}")
        recordset_ids.add(recset)
    recordsets = {rid: {} for rid in recordset_ids}
else:
    print('Record Sets and their fields:')
    for rs_id, rs in recordsets.items():
        print(f'\nRecordSet @id: {rs_id}')
        # List fields if available
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            field_ids = [fields['@id']]
        else:
            field_ids = [field['@id'] for field in fields] if fields else []
        print(f'  Fields: {field_ids}')
        # List columns if given
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            column_ids = [columns['@id']]
        else:
            column_ids = [col['@id'] for col in columns] if columns else []
        print(f'  Columns: {column_ids}')
        # Show the record set's description if any
        desc = rs.get('description', None)
        if desc:
            print(f'  Description: {desc}')

## 3. Data Extraction
Load the data for one or more record sets listed above into pandas DataFrames for analysis, referring to each by its `@id`.

Here, we load a record set (most often there is only one - usually named with "protein", "experiment", or similar in its `@id`). For mass spectrometry data, all the protein expression/quantification data is in a primary table (record set).

In [ ]:
# Get the list of all available record set IDs
available_record_sets = list(dataset.list_record_sets())
print("Available record set @ids:", available_record_sets)

# We'll pick the first one as main (adjust if more exist)
main_record_set_id = available_record_sets[0]

dataframes = {}
for record_set_id in available_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print("Loaded DataFrame for RecordSet @id:", record_set_id)
    print("Columns (by @id):", df.columns.tolist())
    print(df.head(3), "\n")

# For convenience, set df to the first/main record set
df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic data operations:

- Filter proteins/records by abundance or peptide count (fields with numeric data)
- Normalize a selected numeric column
- Group or aggregate by annotation or category

Field and column `@id`s are used to reference each attribute. Adjust `numeric_field_id` as appropriate.

In [ ]:
# Display all columns
print("Available fields by @id:")
for col in df.columns:
    print(col)

# Pick a numeric field (e.g. normalized_abundance, peptide_count, or similar)
# Inspect columns to choose the right one. For this dataset, common options:
#   - 'normalized_abundance@column' or similar (replace if different)

numeric_field_id = None
# Try to guess a good numeric field
candidate_fields = [c for c in df.columns if any(y in c.lower() for y in ['abundance', 'peptide', 'coverage', 'mw', 'count', 'score'])]
for f in candidate_fields:
    if pd.api.types.is_numeric_dtype(df[f]):
        numeric_field_id = f
        break
if not numeric_field_id and candidate_fields:
    # Try to convert to numeric if possible
    f = candidate_fields[0]
    df[f] = pd.to_numeric(df[f], errors='coerce')
    if pd.api.types.is_numeric_dtype(df[f]):
        numeric_field_id = f

if numeric_field_id:
    print(f"Selected numeric field: {numeric_field_id}")
else:
    print("No numeric field found. Please check the dataset column list.")

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
    print(f"Filtering for records with {numeric_field_id} > {threshold:.2f}")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head(5))

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field. Guess protein or sample, or look for 'group', 'type', 'species', etc
    group_field = None
    candidate_groups = [c for c in df.columns if any(y in c.lower() for y in ['sample', 'group', 'type', 'species', 'condition', 'id', 'accession', 'protein']) and c != numeric_field_id]
    if candidate_groups:
        group_field = candidate_groups[0]

    if group_field:
        print(f"Grouping by {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(grouped_df.head())
    else:
        print("No suitable grouping field found.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and, if possible, group means.

You can install `matplotlib` or `seaborn` for richer plots.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        means = df.groupby(group_field)[numeric_field_id].mean()
        means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion

- We demonstrated extracting and inspecting FAIR² mass spectrometry protein records using Croissant's `@id` model, loaded with `mlcroissant`.
- Numeric fields (abundance, peptide counts, etc.) can be filtered, normalized, and grouped for biological or technical interpretation.
- This workflow makes reproducible, standards-compliant exploration of open scientific datasets easy and transparent!
